# 통계 노드 설계

###  0. 데이터 불러오기
- 승인통계 목록

In [3]:
from IPython.display import display
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)


path_survey = "./raw_data/(참고1-1)승인통계_메타데이터.xlsx"

df_survey = pd.read_excel(path_survey, skiprows=1)

In [4]:
df_survey.columns

Index(['승인번호', '기관유형', '작성기관', '통계명', '통계종류', '작성형태', '계속여부', '작성주기', '통계분야',
       '승인일자', '1. 통계기본정보', '1. 작업번호', '2. 통계활용정보', '2. 작업번호', '수집여부', '파일유형',
       '수집여부.1', '파일유형.1', '수집여부.2', '파일유형.2', '해당여부', '파일유형.3'],
      dtype='object')

In [ ]:
df_survey["통계분야"].unique()

array(['범죄·안전', '보건', '과학·기술', '지역통계', '국토이용', '문화.여가', '경제일반·경기', '복지',
       '정보통신', '소득·소비·자산', '에너지', '무역·국제수지', '수산', '농림', '사회일반', '금융',
       '인구', '교육·훈련', '기업경영', '노동', '건설', '교통·물류', '환경', '도소매·서비스',
       '정부·재정', '주거', '국민계정', '임금', '광업·제조업', '물가'], dtype=object)

### 1. 속성 선택
- 수집여부 = 이용자용통계보고서 수집여부
- 수집여부.1 = 간행물 수집여부
- 수집여부.2 = 보도자료 수집여부
- 해당여부 = 통계메타자료 수집여부

In [14]:
df_node = df_survey[["승인번호","통계명","기관유형","작성기관","통계분야","수집여부","수집여부.1","수집여부.2","해당여부"]]

df_node.head()

,승인번호,통계명,기관유형,작성기관,통계분야,수집여부,수집여부.1,수집여부.2,해당여부
0,169003,방위산업실태조사,중앙행정기관,방위사업청,범죄·안전,X,X,X,-
1,117112,장기등기증및이식통계,중앙행정기관,보건복지부,보건,X,X,X,-
2,127022,이공계석박사추적조사,중앙행정기관,과학기술정보통신부,과학·기술,X,X,X,-
3,652005,원주시사회보장수급대상통계,지방자치단체,강원특별자치도 원주시,지역통계,X,X,X,△
4,702005,홍성군청년통계,지방자치단체,충청남도 홍성군,지역통계,X,O,X,-


In [16]:
print(df_node["수집여부"].unique())
print(df_node["수집여부.1"].unique())
print(df_node["수집여부.2"].unique())
print(df_node["해당여부"].unique())

['X' 'O']
['X' 'O']
['X' 'O' ' X']
['-' '△']


자료 수집여부 -> boolen type

In [17]:
df_node["지역통계"] = df_node["통계분야"] == "지역통계"
df_node["이용자용통계정보보고서"] = df_node["수집여부"] == "O"
df_node["간행물"] = df_node["수집여부.1"] == "O"
df_node["보도자료"] = df_node["수집여부.2"] == "O"
df_node["통계설명메타"] = df_node["해당여부"] == "△"

df_node = df_node.drop(columns=["수집여부","수집여부.1","수집여부.2","해당여부"])

df_node.head()

,승인번호,통계명,기관유형,작성기관,통계분야,지역통계,이용자용통계정보보고서,간행물,보도자료,통계설명메타
0,169003,방위산업실태조사,중앙행정기관,방위사업청,범죄·안전,False,False,False,False,False
1,117112,장기등기증및이식통계,중앙행정기관,보건복지부,보건,False,False,False,False,False
2,127022,이공계석박사추적조사,중앙행정기관,과학기술정보통신부,과학·기술,False,False,False,False,False
3,652005,원주시사회보장수급대상통계,지방자치단체,강원특별자치도 원주시,지역통계,True,False,False,False,True
4,702005,홍성군청년통계,지방자치단체,충청남도 홍성군,지역통계,True,False,True,False,False


### 2. 수집등급 부여

- 이용자용통계정보보고서, 간행물, 보도자료, 통계설명메타 수집여부에 따라 중요도 상이
- 이용자용통계정보보고서와 간행물 둘 다 수집하지 못한 경우에만 통계설명메타를 수집한다.

- 1등급 : 보고서 O, 간행물 O
- 2등급 : 보고서 O, 간행물 X
- 3등급 : 보고서 X, 간행물 O
- 4등급 : 보고서 X, 간행물 X

In [18]:
'''
case 1	O	O	O	-	best
case 2	O	O	X	-	very good
case 3	O	X	O	-	good
case 4	O	X	X	-	good
case 5	X	O	O	-	soso
case 6	X	O	X	-	not bad
case 7	X	X	O	O	bad
case 8	X	X	X	O	worst
'''

unique_rows = df_node[["이용자용통계정보보고서", "간행물", "보도자료", "통계설명메타"]].drop_duplicates()

print(unique_rows)
print(f"\n총 고유 조합 개수: {len(unique_rows)}")


     이용자용통계정보보고서    간행물   보도자료  통계설명메타
0          False  False  False   False
3          False  False  False    True
4          False   True  False   False
10         False  False   True    True
20         False   True   True   False
42         False  False   True   False
186         True   True   True   False
192         True   True  False   False
193         True  False   True   False
207         True  False  False   False

총 고유 조합 개수: 10


In [19]:
def assign_level(row):
    if row["이용자용통계정보보고서"] and row["간행물"]:
        return 1
    elif row["이용자용통계정보보고서"] and not row["간행물"]:
        return 2
    elif not row["이용자용통계정보보고서"] and row["간행물"]:
        return 3
    else:
        return 4

df_node["수집등급"] = df_node.apply(assign_level, axis=1)

df_node.head()

,승인번호,통계명,기관유형,작성기관,통계분야,지역통계,이용자용통계정보보고서,간행물,보도자료,통계설명메타,수집등급
0,169003,방위산업실태조사,중앙행정기관,방위사업청,범죄·안전,False,False,False,False,False,4
1,117112,장기등기증및이식통계,중앙행정기관,보건복지부,보건,False,False,False,False,False,4
2,127022,이공계석박사추적조사,중앙행정기관,과학기술정보통신부,과학·기술,False,False,False,False,False,4
3,652005,원주시사회보장수급대상통계,지방자치단체,강원특별자치도 원주시,지역통계,True,False,False,False,True,4
4,702005,홍성군청년통계,지방자치단체,충청남도 홍성군,지역통계,True,False,True,False,False,3


### 3. 통계분야 재분류
- 지방자치단체에서 시행한 통계는 통계분야가 "지역통계"로 분류되었음
- 1333개 중 653개에 해당하므로 통계명에서 지역명을 제거한 다음 "지역통계"를 제외한 나머지 통계분야 중 하나로 분류

행정표준코드관리시스템 > 법정동코드목록조회 > 총 20555건

https://www.code.go.kr/stdcode/regCodeL.do

In [60]:
df_region = df_node[df_node["지역통계"]==True][["승인번호","통계분야","통계명"]]
df_region

,승인번호,통계분야,통계명
3,652005,지역통계,원주시사회보장수급대상통계
4,702005,지역통계,홍성군청년통계
5,701003,지역통계,청양군청년통계
6,741002,지역통계,화순군청년통계
7,734005,지역통계,나주시청년통계
...,...,...,...
1192,204001,지역통계,주민등록인구통계
1193,203001,지역통계,주민등록인구통계
1194,202002,지역통계,주민등록인구통계
1196,217001,지역통계,경상남도사회조사


In [61]:
df_txt = pd.read_csv("./raw_data/법정동코드 전체자료.txt", sep="\t", encoding="cp949")
df_txt.head()

,법정동코드,법정동명,폐지여부
0,1100000000,서울특별시,존재
1,1111000000,서울특별시 종로구,존재
2,1111010100,서울특별시 종로구 청운동,존재
3,1111010200,서울특별시 종로구 신교동,존재
4,1111010300,서울특별시 종로구 궁정동,존재


In [62]:
region_set = set()

for name in df_txt["법정동명"].values:
    parts = name.strip().split()
    if len(parts) == 1:
        region_set.add(parts[0])
    elif len(parts) >= 2:
        if parts[0].endswith("시"):
            region_set.add(parts[0])
        else:
            region_set.add(parts[0])
            region_set.add(parts[1])

print(region_set)

{'김천시', '양주군', '온양시', '구례군', '보성군', '계룡시', '신안동시', '영풍군', '경기도', '정읍시', '울산시', '해남군', '광산군', '순창군', '홍천군', '화성군', '구미시', '선산군', '영천시', '이천시', '대덕군', '광양시', '신안군', '인천광역시', '안동시', '평택시', '울산군', '삼천포시', '단양군', '정선군', '철원군', '제원군', '완주군', '나주군', '동광양시', '용인시', '태백시', '달성군', '마산시', '무주군', '대구광역시', '오산시', '당진군', '칠곡군', '함안군', '여천시', '서귀포시', '김제군', '영동군', '광명시', '연천군', '목포시', '수원시', '상주군', '상주시', '고흥군', '당진시', '영일군', '송탄시', '천안시', '무안군', '남양주시', '김제시', '양평군', '경상남도', '의정부시', '포천군', '경주군', '봉화군', '진천군', '옹진군', '남양주군', '경산군', '증평군', '논산시', '문경군', '화천군', '금릉군', '제주도', '천원군', '강원특별자치도', '미금시', '중원군', '인천직할시', '점촌시', '영광군', '명주군', '안양시', '울진군', '강원도', '평택군', '춘성군', '성남시', '사천군', '진해시', '거제시', '남제주군', '삼척군', '남원군', '문경시', '통영군', '제주특별자치도', '함평군', '성주군', '용인군', '옥천군', '의왕시', '진안군', '이리시', '남원시', '장흥군', '장수군', '김포시', '예천군', '창원시', '광주광역시', '김포군', '서산시', '전라북도', '순천시', '영덕군', '원주시', '양양군', '금산군', '나주시', '광주직할시', '영주시', '시흥시', '고성군', '진주시', '충청남도', '의령군', '동래군', '사천시', '원주군', '양산시', '남해군', '포천시', '여주군

In [63]:
do_set = set()
si_set = set()
gun_set = set()

for region in region_set:
    if region.endswith("도"):
        do_set.add(region)
    elif region.endswith("시"):
        si_set.add(region)
    else:
        gun_set.add(region)

# 출력
print("✅ 도:")
print(sorted(do_set))

print("\n✅ 시:")
print(sorted(si_set))

print("\n✅ 군:")
print(sorted(gun_set))


✅ 도:
['강원도', '강원특별자치도', '경기도', '경상남도', '경상북도', '전라남도', '전라북도', '전북특별자치도', '제주도', '제주특별자치도', '충청남도', '충청북도']

✅ 시:
['강릉시', '거제시', '경산시', '경주시', '계룡시', '고양시', '공주시', '과천시', '광명시', '광양시', '광주광역시', '광주시', '광주직할시', '구리시', '구미시', '군산시', '군포시', '금성시', '김제시', '김천시', '김포시', '김해시', '나주시', '남양주시', '남원시', '논산시', '당진시', '대구광역시', '대구시', '대구직할시', '대전광역시', '대전시', '대전직할시', '대천시', '동광양시', '동두천시', '동해시', '마산시', '목포시', '문경시', '미금시', '밀양시', '보령시', '부산광역시', '부산시', '부산직할시', '부천시', '사천시', '삼척시', '삼천포시', '상주시', '서귀포시', '서산시', '서울특별시', '성남시', '세종특별자치시', '속초시', '송정시', '송탄시', '수원시', '순천시', '시흥시', '신안동시', '아산시', '안동시', '안산시', '안성시', '안양시', '양산시', '양주시', '여수시', '여주시', '여천시', '영주시', '영천시', '오산시', '온양시', '용인시', '울산광역시', '울산시', '원주시', '의왕시', '의정부시', '이리시', '이천시', '익산시', '인천광역시', '인천시', '인천직할시', '장승포시', '전주시', '점촌시', '정읍시', '정주시', '제주시', '제천시', '진주시', '진해시', '창원시', '천안시', '청주시', '춘천시', '충무시', '충주시', '태백시', '통영시', '파주시', '평택시', '포천시', '포항시', '하남시', '화성시']

✅ 군:
['가평군', '강진군', '강화군', '거제군', '거창군', '경산군', '경주군', '고령군', '고

지역명 제거

In [64]:
import re

# 지역명을 정규식 패턴으로 묶음 (ex. '서울특별시|경기도|강남구|...')
region_pattern = "|".join(sorted(region_set, key=len, reverse=True))  # 긴 이름부터 우선 매치

# 정규표현식을 활용해 지역명 제거
df_region.insert(df_region.columns.get_loc("통계명") + 1,
                 "정제된 통계명",
                 df_region["통계명"].str.replace(region_pattern, "", regex=True).str.strip())

df_region = df_region.reset_index(drop=True) 

df_region

,승인번호,통계분야,통계명,정제된 통계명
0,652005,지역통계,원주시사회보장수급대상통계,사회보장수급대상통계
1,702005,지역통계,홍성군청년통계,청년통계
2,701003,지역통계,청양군청년통계,청년통계
3,741002,지역통계,화순군청년통계,청년통계
4,734005,지역통계,나주시청년통계,청년통계
...,...,...,...,...
648,204001,지역통계,주민등록인구통계,주민등록인구통계
649,203001,지역통계,주민등록인구통계,주민등록인구통계
650,202002,지역통계,주민등록인구통계,주민등록인구통계
651,217001,지역통계,경상남도사회조사,사회조사


새로운 통계분야로 분류 (17분 소요)

In [65]:
import os
import yaml
import time
from dotenv import load_dotenv
from openai import OpenAI


# 🔹 CLOVA API, URL 로드
load_dotenv()
CLOVA_API_KEY = os.getenv("CLOVASTUDIO_API_KEY")
BASE_URL = os.getenv("CLOVASTUDIO_API_BASE_URL")


# 🔹 프롬프트 템플릿 로드
with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)
PROMPT_TEMPLATE = config["survey_classification_template"]


# 🔹 OpenAI 호환 client 구성
client = OpenAI(
    api_key=CLOVA_API_KEY,
    base_url=BASE_URL
)
def execute_clovastudio(user_input: str):
    prompt = PROMPT_TEMPLATE.format(survey_name=user_input)

    response = client.chat.completions.create(
        model="HCX-005",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    time.sleep(0.5)
    return response.choices[0].message.content

In [66]:
user_input = df_region["정제된 통계명"].values[0]
result = execute_clovastudio(user_input)

result

'[분류]: 복지'

In [67]:
df_region["재분류결과"] = df_region["정제된 통계명"].apply(execute_clovastudio)

In [70]:
df_region["재분류결과"] = df_region["재분류결과"].str.replace(r'^\[분류\]:\s*', '', regex=True)

In [71]:
df_region

,승인번호,통계분야,통계명,정제된 통계명,재분류결과
0,652005,지역통계,원주시사회보장수급대상통계,사회보장수급대상통계,복지
1,702005,지역통계,홍성군청년통계,청년통계,사회일반
2,701003,지역통계,청양군청년통계,청년통계,사회일반
3,741002,지역통계,화순군청년통계,청년통계,사회일반
4,734005,지역통계,나주시청년통계,청년통계,사회일반
...,...,...,...,...,...
648,204001,지역통계,주민등록인구통계,주민등록인구통계,인구
649,203001,지역통계,주민등록인구통계,주민등록인구통계,인구
650,202002,지역통계,주민등록인구통계,주민등록인구통계,인구
651,217001,지역통계,경상남도사회조사,사회조사,사회일반


전체 데이터에 변경사항 적용

In [74]:
df_node_survey = df_node.copy()

reclassification_dict = df_region.set_index("승인번호")["재분류결과"]

df_node_survey["통계분야"] = df_node["승인번호"].map(reclassification_dict).fillna(df_node["통계분야"])

df_node_survey

,승인번호,통계명,기관유형,작성기관,통계분야,지역통계,이용자용통계정보보고서,간행물,보도자료,통계설명메타,수집등급
0,169003,방위산업실태조사,중앙행정기관,방위사업청,범죄·안전,False,False,False,False,False,4
1,117112,장기등기증및이식통계,중앙행정기관,보건복지부,보건,False,False,False,False,False,4
2,127022,이공계석박사추적조사,중앙행정기관,과학기술정보통신부,과학·기술,False,False,False,False,False,4
3,652005,원주시사회보장수급대상통계,지방자치단체,강원특별자치도 원주시,복지,True,False,False,False,True,4
4,702005,홍성군청년통계,지방자치단체,충청남도 홍성군,사회일반,True,False,True,False,False,3
...,...,...,...,...,...,...,...,...,...,...,...
1328,101041,농림어업총조사,중앙행정기관,통계청,농림,False,True,True,True,False,1
1329,101004,경제활동인구조사,중앙행정기관,통계청,노동,False,True,False,True,False,2
1330,101003,인구동향조사,중앙행정기관,통계청,인구,False,True,False,True,False,2
1331,101002,주택총조사,중앙행정기관,통계청,주거,False,True,True,True,False,1


In [76]:
df_node_survey.to_csv("./nodes/node_survey.csv", index=False, encoding="utf-8-sig")